# StatsBomb Open Data

Free, publicly released football event data:
[github.com/statsbomb/open-data](https://github.com/statsbomb/open-data)

Downloaded and processed for this project: 8,536 shots from 350
professional matches across eight competitions.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from pitchsim.statsbomb import SHOTS_PARQUET, fetch_events

print("loading from:", SHOTS_PARQUET.relative_to(ROOT))
shots = pd.read_parquet(SHOTS_PARQUET)
print(f"  {len(shots):,} shots from {shots.match_id.nunique()} matches")

loading from: data\statsbomb\shots.parquet


  8,536 shots from 350 matches


## 1. What the dataset contains

StatsBomb records every event in a match: passes, shots, fouls,
goalkeeper actions, substitutions. Below is a single real match,
downloaded live from their public repository.

In [2]:
events = fetch_events(3773386, cache=False)   # Barcelona vs Deportivo Alaves

print(f"events in this one match: {len(events):,}\n")
for name, count in Counter(e["type"]["name"] for e in events).most_common(14):
    print(f"  {name:22} {count:>5}")

print("\nWe extract the Shot events only - roughly 30 per match,")
print("which across 350 matches gives the 8,536 rows used here.")

events in this one match: 3,891

  Pass                    1118
  Ball Receipt*           1059
  Carry                    888
  Pressure                 307
  Ball Recovery             99
  Duel                      63
  Dribble                   40
  Block                     39
  Clearance                 39
  Goal Keeper               35
  Dispossessed              29
  Shot                      29
  Dribbled Past             26
  Miscontrol                26

We extract the Shot events only - roughly 30 per match,
which across 350 matches gives the 8,536 rows used here.


## 2. A raw shot event

One shot, exactly as StatsBomb publishes it. Note there is no distance
or angle here, only a pitch coordinate and a list of player positions.

In [3]:
shot = [e for e in events if e["type"]["name"] == "Shot"][4]

print(json.dumps(shot, indent=2)[:1800])

{
  "id": "75bdc651-c041-4021-b201-cb9eb8b97837",
  "index": 1282,
  "period": 1,
  "timestamp": "00:30:06.585",
  "minute": 30,
  "second": 6,
  "type": {
    "id": 16,
    "name": "Shot"
  },
  "possession": 52,
  "possession_team": {
    "id": 217,
    "name": "Barcelona"
  },
  "play_pattern": {
    "id": 1,
    "name": "Regular Play"
  },
  "team": {
    "id": 206,
    "name": "Deportivo Alav\u00e9s"
  },
  "player": {
    "id": 24049,
    "name": "Luis Jes\u00fas Rioja Gonz\u00e1lez"
  },
  "position": {
    "id": 16,
    "name": "Left Midfield"
  },
  "location": [
    118.3,
    42.1
  ],
  "duration": 0.24149,
  "related_events": [
    "0c19174e-aa16-4b56-ab40-9b29e27baa1d"
  ],
  "shot": {
    "one_on_one": true,
    "open_goal": true,
    "statsbomb_xg": 0.9761923,
    "end_location": [
      120.0,
      40.7,
      0.0
    ],
    "technique": {
      "id": 93,
      "name": "Normal"
    },
    "body_part": {
      "id": 38,
      "name": "Left Foot"
    },
    "type": {
  

## 3. After our processing

Shot distance, the angle the goal subtends, how many defenders stand in
the shooting cone and where the goalkeeper is are all computed by us from
the raw coordinates and freeze frame above.

In [4]:
columns = [
    "distance_m", "angle_rad", "is_header", "under_pressure",
    "defenders_in_cone", "nearest_defender_m", "gk_off_line_m",
    "goal", "sb_xg",
]
shots[columns].head(10).round(3)

,distance_m,angle_rad,is_header,under_pressure,defenders_in_cone,nearest_defender_m,gk_off_line_m,goal,sb_xg
0,14.267,0.367,0,0,0.0,2.493,4.812,0,0.201
1,17.127,0.359,0,0,0.0,5.558,3.238,0,0.096
2,14.683,0.461,0,0,1.0,2.380,1.312,0,0.099
3,19.680,0.362,0,0,7.0,8.951,0.350,0,0.079
4,2.324,2.204,0,0,0.0,14.001,8.488,1,0.976
5,6.067,1.084,1,1,0.0,0.551,1.050,0,0.203
6,16.419,0.437,0,0,6.0,8.104,0.788,0,0.095
7,30.939,0.235,0,0,2.0,3.853,0.875,0,0.018
8,14.208,0.463,0,1,0.0,0.351,1.488,0,0.112
9,13.190,0.261,0,0,0.0,0.971,1.575,0,0.054


## 4. Summary

In [5]:
n_matches = shots.match_id.nunique()

print(f"shots             : {len(shots):,}")
print(f"matches           : {n_matches}")
print(f"goals             : {int(shots.goal.sum()):,}")
print(f"conversion rate   : {shots.goal.mean():.1%}")
print(f"shots per match   : {len(shots) / n_matches:.1f}")
print(f"mean StatsBomb xG : {shots.sb_xg.mean():.3f}")
print(f"with freeze frame : {shots.has_freeze_frame.mean():.1%}")
print()
print("competitions:")
for name in ["La Liga 2020/21", "La Liga 2019/20", "Premier League 2003/04",
             "Bundesliga 2023/24", "UEFA Euro 2020", "FIFA World Cup 2022",
             "Women's World Cup 2023", "UEFA Women's Euro 2022"]:
    print(f"  {name}")

shots             : 8,536
matches           : 350
goals             : 871
conversion rate   : 10.2%
shots per match   : 24.4
mean StatsBomb xG : 0.099
with freeze frame : 100.0%

competitions:
  La Liga 2020/21
  La Liga 2019/20
  Premier League 2003/04
  Bundesliga 2023/24
  UEFA Euro 2020
  FIFA World Cup 2022
  Women's World Cup 2023
  UEFA Women's Euro 2022
